In [ ]:
import os
from glob import glob
import numpy as np
import pandas as pd
import rioxarray

import dask
import dask.dataframe as dd
from dask.distributed import Client


# 合并逐日数据，输出大洲组合数据
- 这个计算时间很长（1小时以上），因为会读取每一个daily文件（这里还读了两遍）
- 将几个大洲的数据合并输出，是为了减少计算时间（这里分成了1-4，5-9两批，所以读两遍；如果逐大洲输出，就需要读9遍）
- 但也要考虑电脑内存量：如果内存不够大，需要分成更多批次
- 大洲代号：1-af,2-eu,3-si,4-as,5-au,6-sa,7-na,8-ar,9-gr

In [2]:
dates=pd.date_range(start='2023-07-26',end='2024-07-22')# 处理的日期范围
date_strings=dates.strftime("%Y%m%d").tolist()

In [ ]:

name_list=[['af','eu','si','as'],['au','sa','na','ar','gr']]
for j in range(2):
    dfs=[]
    for date in date_strings:
        file_path='D:/HC_P_data/RiverSP_s/cycle1/node_'+date+'.csv'
        if os.path.exists(file_path):
            df_merge=pd.read_csv(file_path)
            df_merge=df_merge[df_merge['wse']>-1e5]# 去除wse的无效数据

            # 获取node的大洲信息：reach_id的第一个数据
            df_merge['reach_id']=df_merge['node_id']//10000*10+df_merge['node_id']%10
            df_merge['continent']=df_merge['reach_id']//1e10
            # 分成两批
            if j==0:
                df_merge=df_merge[df_merge['continent']<=4]# continent:1-4
            else:
                df_merge=df_merge[df_merge['continent']>4]# continent:5-9
            
            # 整理需要保留的列
            df_merge=df_merge[['node_id','time','wse','wse_u','width','ice_clim_f','dark_frac','rdr_sig0','node_q_b','xovr_cal_q']]
            if df_merge.empty:
                continue
            dfs.append(df_merge)
            print(date)
    df=pd.concat(dfs)
    if j==0:
        df.to_csv('D:/HC_P_data/RiverSP_s/cycle1/node_1-4.csv')
    else:
        df.to_csv('D:/HC_P_data/RiverSP_s/cycle1/node_5-9.csv')

# 拆分大洲组合数据，变成逐大洲数据

In [ ]:
name_list = [['af', 'eu', 'si', 'as'], ['au', 'sa', 'na', 'ar', 'gr']]
# 定义路径和相关变量
input_file = 'D:/HC_P_data/RiverSP_s/cycle1/node_1-4_v3.csv'
output_dir = 'D:/HC_P_data/RiverSP_s/cycle1/'

# 使用 Dask 读取 CSV 文件（并行计算，减少计算时间）
df = dd.read_csv(input_file)

# 计算 reach_id 和 continent
df['reach_id'] = df['node_id'] // 10000 * 10 + df['node_id'] % 10
df['continent'] = (df['reach_id'] // 1e10).astype(int)

# 根据 continent 分别保存为不同的文件
for i in range(4):  # continent 是 1 到 4
    continent_df = df[df['continent'] == (i + 1)]
    output_file = f"{output_dir}node_{name_list[0][i]}.csv"
    continent_df.to_csv(output_file, single_file=True, index=False)  # 保存到文件
    print(f"Continent {i + 1} saved to {output_file}")

In [ ]:
# 第二个文件
input_file = 'D:/HC_P_data/RiverSP_s/cycle1/node_5-9_v3.csv'

# 使用 Dask 读取 CSV 文件
df = dd.read_csv(input_file)

# 计算 reach_id 和 continent
df['reach_id'] = df['node_id'] // 10000 * 10 + df['node_id'] % 10
df['continent'] = (df['reach_id'] // 1e10).astype(int)

# 根据 continent 分别保存为不同的文件
for i in range(5):  # continent 是 5 到 9
    continent_df = df[df['continent'] == (i + 5)]
    output_file = f"{output_dir}node_{name_list[1][i]}.csv"
    continent_df.to_csv(output_file, single_file=True, index=False)  # 保存到文件
    print(f"Continent {i + 1} saved to {output_file}")

# sigma0的年平均

In [ ]:
%%time
dfs = dd.read_csv('D:/HC_P_data/RiverSP_s/cycle1/node_*.csv')
grouped = dfs.groupby('node_id')['rdr_sig0'].mean()

CPU times: total: 15.6 ms
Wall time: 12 ms


In [ ]:
%%time
result = grouped.compute()

CPU times: total: 14min 9s
Wall time: 2min 46s


In [ ]:
result.to_csv('D:/HC_P_data/RiverSP_s/cycle1/node_sig_mean.csv')

# water occurrence数据
- 这个原始数据下载和处理太麻烦，处理代码放在这里，可以进行recheck
- OSW的原始数据：https://global-surface-water.appspot.com/download
- 直接下载node_geometry文件（输入数据）（超过100M，不能通过github同步）：
通过网盘分享的文件：node_geometry.csv
链接: https://pan.baidu.com/s/1Clox60OHBTWQrFmFA3_uHA 提取码: 6zw1 
--来自百度网盘超级会员v6的分享
- 直接下载Water occurrence的结果（输出数据）：
通过网盘分享的文件：node_occu_mean.csv
链接: https://pan.baidu.com/s/1Ba9eNJ_Ep3NA4rh6JGyumA 提取码: c3di 
--来自百度网盘超级会员v6的分享

In [ ]:
client = Client(dashboard_address=':3894')

In [ ]:
def calculate_mean(row, ds):
    # Select the nearest pixel
    nearest_pixel = ds.sel(x=row['lon'], y=row['lat'], method="nearest")
    # Get the indices of the nearest pixel
    x_idx = np.abs(ds.x - nearest_pixel.x).argmin().item()
    y_idx = np.abs(ds.y - nearest_pixel.y).argmin().item()
    print(x_idx)
    # Slice a 5x5 area around the nearest pixel
    area = ds.isel(x=slice(x_idx - 2, x_idx + 3), 
                  y=slice(y_idx - 2, y_idx + 3))
    # Calculate the mean value of the area
    return area.mean().values

def process_file(file_path, df):
    ds = rioxarray.open_rasterio(file_path)
    in_bounds = (df['lon'] > ds.x.min().values) & (df['lon'] < ds.x.max().values) & (df['lat'] > ds.y.min().values) & (df['lat'] < ds.y.max().values)
    df_sub = df[in_bounds].copy()
    
    if df_sub.empty:
        return pd.DataFrame(columns=['node_id', 'occurrence'])
    
    df_sub['occurrence'] = df_sub.apply(calculate_mean, axis=1, ds=ds)
    return df_sub[['node_id', 'occurrence']]

def parallel_processing(file_paths, df):
    delayed_tasks = []
    
    for file_path in file_paths:
        delayed_task = dask.delayed(process_file)(file_path, df)
        delayed_tasks.append(delayed_task)
        
    # Execute delayed tasks and collect results
    dfs = dask.compute(*delayed_tasks)
    return dfs


In [ ]:
# Load the DataFrame
df = pd.read_csv('D:/HC_P_data/csv_result/node_geometry.csv')# 前面给了下载链接

# Convert to Dask DataFrame
df = dd.from_pandas(df)#, npartitions=4)  # Adjust the number of partitions as needed

# Get file paths
file_paths = glob('D:/HC_P_data/GSW/tiles/occurrence-*.tif')# 这个14.8GB，太大了，上传不了

# Call parallel processing function
dfs = parallel_processing(file_paths, df)

# Concatenate results
result_df = pd.concat(dfs)
result_df.to_csv('D:/HC_P_data/RiverSP_s/node_occu_mean.csv')# 给下载链接

In [ ]:
client.close()